In [1]:
import joblib

X_train = joblib.load(
    "X_train_random.pkl"
)

X_test = joblib.load(
    "X_test_random.pkl"
)

y_train = joblib.load(
    "y_train_random.pkl"
)

y_test = joblib.load(
    "y_test_random.pkl"
)

In [2]:
import numpy as np
import pandas as pd
import optuna

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import RobustScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_regression
)

from sklearn.model_selection import cross_validate

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from xgboost import XGBRegressor

In [3]:
preprocessor_robust = Pipeline([

    (
        "imputer",
        SimpleImputer(strategy="mean")
    ),

    (
        "variance_filter",
        VarianceThreshold(threshold=0.01)
    ),

    (
        "scaler",
        RobustScaler()
    ),

    (
        "feature_selection",
        SelectKBest(
            score_func=mutual_info_regression,
            k=1000
        )
    )

])

In [4]:

def objective(trial):

    model = XGBRegressor(

        n_estimators=trial.suggest_int(
            "n_estimators",
            200,
            1200
        ),

        max_depth=trial.suggest_int(
            "max_depth",
            3,
            12
        ),

        learning_rate=trial.suggest_float(
            "learning_rate",
            0.01,
            0.3,
            log=True
        ),

        min_child_weight=trial.suggest_int(
            "min_child_weight",
            1,
            20
        ),

        subsample=trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        colsample_bytree=trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),

        gamma=trial.suggest_float(
            "gamma",
            0,
            5
        ),

        reg_alpha=trial.suggest_float(
            "reg_alpha",
            0,
            10
        ),

        reg_lambda=trial.suggest_float(
            "reg_lambda",
            0,
            10
        ),

        objective="reg:squarederror",

        random_state=42,

        n_jobs=1
    )

    pipeline = Pipeline([

        (
            "preprocessor_robust",
            preprocessor_robust
        ),

        (
            "model",
            model
        )

    ])

    scores = cross_validate(

        pipeline,

        X_train,

        y_train,

        cv=5,

        scoring={

            "r2": "r2",

            "mae": "neg_mean_absolute_error"

        },

        n_jobs=-1

    )

    mean_r2 = np.mean(
        scores["test_r2"]
    )

    mean_mae = -np.mean(
        scores["test_mae"]
    )

    trial.set_user_attr(
        "mean_r2",
        mean_r2
    )

    trial.set_user_attr(
        "mean_mae",
        mean_mae
    )

    print(
        f"Trial {trial.number} | "
        f"Mean R2 = {mean_r2:.4f} | "
        f"Mean MAE = {mean_mae:.4f}"
    )

    return mean_r2

In [5]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=30,
    show_progress_bar=True
)

[I 2026-06-08 21:53:18,333] A new study created in memory with name: no-name-52da25ff-2d81-4772-9fd4-3f0e6ee480ac


  0%|          | 0/30 [00:00<?, ?it/s]

Trial 0 | Mean R2 = 0.7867 | Mean MAE = 0.7329
[I 2026-06-08 21:58:29,950] Trial 0 finished with value: 0.7866564424759588 and parameters: {'n_estimators': 664, 'max_depth': 11, 'learning_rate': 0.017988023607854575, 'min_child_weight': 7, 'subsample': 0.9409476155887324, 'colsample_bytree': 0.6567227469791043, 'gamma': 1.967550554548707, 'reg_alpha': 4.624124058363435, 'reg_lambda': 9.562469467210025}. Best is trial 0 with value: 0.7866564424759588.
Trial 1 | Mean R2 = 0.7622 | Mean MAE = 0.7750
[I 2026-06-08 22:01:58,352] Trial 1 finished with value: 0.7622146214988673 and parameters: {'n_estimators': 735, 'max_depth': 10, 'learning_rate': 0.2533356399180629, 'min_child_weight': 11, 'subsample': 0.7257238085256493, 'colsample_bytree': 0.7141080477803827, 'gamma': 3.0885254962633213, 'reg_alpha': 1.6548124642462958, 'reg_lambda': 6.420215333018417}. Best is trial 0 with value: 0.7866564424759588.
Trial 2 | Mean R2 = 0.7860 | Mean MAE = 0.7370
[I 2026-06-08 22:06:48,263] Trial 2 finish

In [8]:
print("Best Mean CV R2:")
print(study.best_value)

print("\nBest Parameters:")
print(study.best_params)

Best Mean CV R2:
0.790765202783965

Best Parameters:
{'n_estimators': 1029, 'max_depth': 6, 'learning_rate': 0.023156918649660435, 'min_child_weight': 3, 'subsample': 0.8145729287432204, 'colsample_bytree': 0.711184224184241, 'gamma': 1.3806200387979817, 'reg_alpha': 4.103337341071372, 'reg_lambda': 7.977824721362305}


In [9]:
best_xgb_pipeline = Pipeline([

    (
        "preprocessor_robust",
        preprocessor_robust
    ),

    (
        "model",

        XGBRegressor(

            **study.best_params,

            objective="reg:squarederror",

            random_state=42,

            n_jobs=1
        )
    )

])

In [10]:
best_xgb_pipeline.fit(
    X_train,
    y_train
)

,steps,"[('preprocessor_robust', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,steps,"[('imputer', ...), ('variance_filter', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'mean'
,fill_value,None


In [11]:
y_train_pred = best_xgb_pipeline.predict(
    X_train
)

y_test_pred = best_xgb_pipeline.predict(
    X_test
)

In [12]:
train_r2 = r2_score(
    y_train,
    y_train_pred
)

test_r2 = r2_score(
    y_test,
    y_test_pred
)

train_rmse = np.sqrt(
    mean_squared_error(
        y_train,
        y_train_pred
    )
)

test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

train_mae = mean_absolute_error(
    y_train,
    y_train_pred
)

test_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

In [13]:
print("\n TUNED XGBOOST ")

print("\nTrain R2 :", train_r2)
print("Train RMSE :", train_rmse)
print("Train MAE :", train_mae)

print("\nTest R2 :", test_r2)
print("Test RMSE :", test_rmse)
print("Test MAE :", test_mae)


 TUNED XGBOOST 

Train R2 : 0.9294959898683726
Train RMSE : 0.6313331006416515
Train MAE : 0.40044336634860656

Test R2 : 0.8187214148959565
Test RMSE : 0.9913712306509218
Test MAE : 0.644180990178219


In [14]:
import pandas as pd

trials_df = study.trials_dataframe()

trials_df["Mean_CV_MAE"] = [
    t.user_attrs.get("mean_mae")
    for t in study.trials
]

trials_df["Mean_CV_R2"] = [
    t.user_attrs.get("mean_r2")
    for t in study.trials
]

final_trials_df = trials_df[
    [
        "number",
        "Mean_CV_R2",
        "Mean_CV_MAE",
        "params_n_estimators",
        "params_max_depth",
        "params_learning_rate",
        "params_min_child_weight",
        "params_subsample",
        "params_colsample_bytree",
        "params_gamma",
        "params_reg_alpha",
        "params_reg_lambda",
        "state"
    ]
].copy()

final_trials_df.columns = [
    "Trial_Number",
    "Mean_CV_R2",
    "Mean_CV_MAE",
    "N_Estimators",
    "Max_Depth",
    "Learning_Rate",
    "Min_Child_Weight",
    "Subsample",
    "Colsample_Bytree",
    "Gamma",
    "Reg_Alpha",
    "Reg_Lambda",
    "Status"
]

final_trials_df.to_csv(
    "xgb_optuna_trials.csv",
    index=False
)

final_trials_df.head()

,Trial_Number,Mean_CV_R2,Mean_CV_MAE,N_Estimators,Max_Depth,Learning_Rate,Min_Child_Weight,Subsample,Colsample_Bytree,Gamma,Reg_Alpha,Reg_Lambda,Status
0,0,0.786656,0.732888,664,11,0.017988,7,0.940948,0.656723,1.967551,4.624124,9.562469,COMPLETE
1,1,0.762215,0.774984,735,10,0.253336,11,0.725724,0.714108,3.088525,1.654812,6.420215,COMPLETE
2,2,0.785987,0.737029,708,4,0.035528,15,0.762050,0.821936,1.604081,8.573435,4.094315,COMPLETE
3,3,0.779171,0.753100,1117,12,0.083477,15,0.923519,0.835651,3.895419,6.169833,7.277007,COMPLETE
4,4,0.785122,0.736861,959,10,0.020593,18,0.909343,0.800297,3.334755,3.289734,2.788130,COMPLETE


In [15]:
import joblib

joblib.dump(
    best_xgb_pipeline,
    "best_xgb_pipeline.pkl"
)

['best_xgb_pipeline.pkl']

In [16]:
metrics_df = pd.DataFrame({

    "Metric": ["R2", "MAE", "RMSE"],

    "Train": [
        train_r2,
        train_mae,
        train_rmse
    ],

    "Test": [
        test_r2,
        test_mae,
        test_rmse
    ]

})

metrics_df.to_csv(
    "xgb_final_metrics.csv",
    index=False
)

In [17]:
xgb = pd.read_csv("xgb_optuna_trials.csv")

xgb.loc[
    xgb["Mean_CV_R2"].idxmax()
]

Trial_Number              27
Mean_CV_R2          0.790765
Mean_CV_MAE         0.716335
N_Estimators            1029
Max_Depth                  6
Learning_Rate       0.023157
Min_Child_Weight           3
Subsample           0.814573
Colsample_Bytree    0.711184
Gamma                1.38062
Reg_Alpha           4.103337
Reg_Lambda          7.977825
Status              COMPLETE
Name: 27, dtype: object

In [18]:
import os

print(os.getcwd())
print(os.listdir())

c:\Users\jeshw\Desktop\99_sol\notebooks
['all_2453_features_with_drugid.csv', 'baseline_1.ipynb', 'baseline_2.ipynb', 'baseline_3.ipynb', 'base_strat_1.ipynb', 'base_strat_2.ipynb', 'base_strat_3.ipynb', 'best_lgbm_pipeline.pkl', 'best_svr_pipeline.pkl', 'best_xgb_pipeline.pkl', 'catboost_info', 'cat_pipeline.pkl', 'descriptor_df', 'final_feature_columns.pkl', 'index.ipynb', 'lgbm_final_metrics.csv', 'lgbm_optuna_trials.csv', 'lgbm_pipeline.pkl', 'lightGBM_random_base_1_cv_tunned.ipynb', 'lightGBM_random_base_1_cv_tunned_2..ipynb', 'lightGBM_random_base_1_tuned.ipynb', 'preprocess_1.ipynb', 'results', 'rf_pipeline.pkl', 'selected_1000_features.pkl', 'selected_1000_features_with_drugid.csv', 'selected_1000_feature_names.csv', 'svr_final_metrics.csv', 'svr_optuna_trials.csv', 'svr_pipeline.pkl', 'svr_stratifed_base_1_tuned_2.ipynb', 'svr_stratifed_base_1_tuned_optun_cv.ipynb', 'xgb_final_metrics.csv', 'xgb_optuna_trials.csv', 'xgb_pipeline', 'xgb_rand_b2.ipynb', 'xgb_rand_b2_cv_tunned.ip